# Full Pipeline — Sparse-Sensor Graph-Aware Faulted-Section Localization

IEEE 33-bus feeder → PV + load variation → line-outage faults → sparse/noisy/missing sensing → dataset → threshold / Random Forest / graph-aware models → restoration comparison.

**Run with `N_SCENARIOS = 300` first.** Only raise it once every cell runs clean.

### Three fixes versus the original skeleton, and why

1. **5 feature channels, not 4.** Originally "no sensor here", "sensor missing", and "bus is de-energised" all encoded as `0.0`, so no model could tell them apart. Now `ch3 = sensor exists`, `ch4 = value valid`.
2. **PV categories vary spread, not mean.** Originally `low` averaged 0.75 output and `high` averaged 0.50 — so a "PV variability" result was really a PV *level* result. Now all three share mean 0.5.
3. **Concat pooling, not mean pooling.** `h.mean(dim=1)` averaged all 33 nodes into one vector, discarding the spatial information the whole hypothesis rests on.

Also: restoration mutates `in_service` flags instead of `deepcopy`-ing the network, which is roughly an order of magnitude faster and lets you actually reach 10,000 scenarios.

In [ ]:
!pip -q install pandapower scikit-learn networkx matplotlib pandas numpy torch

In [ ]:
import copy, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import pandapower as pp
import pandapower.networks as pn
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

N_SCENARIOS   = 2000       # raise to 5000-10000 for the final run

# Voltage safety. The 33-bus base case already sits near 0.91 pu, so a
# fixed 0.95 floor rejects every state including healthy ones. Instead:
# a restoration action is unsafe only if it drives voltage BELOW the
# base case by more than V_TOLERANCE, or above VMAX.
# Defensible in the paper: you are testing whether switching makes the
# feeder worse, not whether the feeder was ever ideal.
VMIN_ABS      = 0.90       # hard floor
V_TOLERANCE   = 0.02       # allowed degradation below base-case minimum
VMAX          = 1.05
MAX_LOADING   = 100.0
FAULT_PROB    = 0.50       # balanced classes; 0.65 let detectors cheat
PV_PENETRATION = 0.40      # total PV capacity as fraction of base load

# Faults are sampled from ALL closed lines.
#
# An earlier attempt restricted faults to lines with no sensor downstream,
# to stop models reading off which sensors went dead. On this feeder that
# pool is empty for the 10- and 14-sensor layouts, because sensors sit at
# the feeder extremities. More importantly, loss-of-voltage IS a genuine
# physical indication that real FLISR schemes rely on -- suppressing it
# would model a grid that does not exist.
#
# The resulting task shape, which you should state plainly in the paper:
#   detection is easy (de-energisation is unmistakable)
#   localization is hard (each section aggregates several lines, and
#   different lines can produce identical dead-sensor patterns, so the
#   model must separate them using voltage magnitudes and topology)
HIDDEN_FAULTS_ONLY = False

SENSOR_LAYOUTS = {
    "S6":  [0, 5, 10, 17, 24, 32],
    "S10": [0, 3, 5, 8, 12, 17, 21, 24, 28, 32],
    "S14": [0, 2, 4, 6, 8, 10, 12, 14, 17, 21, 24, 27, 30, 32],
}
NOISE_LEVELS  = [0.00, 0.01, 0.02, 0.05]
MISSING_PROBS = [0.00, 0.05, 0.10, 0.20]

print("pandapower", pp.__version__, "| torch", torch.__version__)

## 1. Base network + empirical tie-line detection

Ties are found by topology (removing a tie leaves the graph connected), not by assuming they are the last five rows.

In [ ]:
def get_ties(net):
    """Tie-switches from pandapower's own in_service flags.

    An earlier topological detector (remove an edge, keep it if the graph
    stays connected) was WRONG: with all 5 ties present the feeder has 5
    loops, so ordinary sectionalizing branches also pass that test. It
    classified 36 of 37 branches as ties and left a single closed line.

    case33bw ships the 5 ties with in_service=False, so read the flags
    and then VERIFY the remainder is a spanning tree.
    """
    ties = [int(l) for l in net.line.index[net.line["in_service"] == False]]
    closed = [int(l) for l in net.line.index if l not in ties]
    G = nx.Graph()
    G.add_nodes_from(int(b) for b in net.bus.index)
    for l in closed:
        r = net.line.loc[l]
        G.add_edge(int(r["from_bus"]), int(r["to_bus"]))
    assert nx.is_connected(G), "closed lines do not span all buses"
    assert G.number_of_edges() == G.number_of_nodes() - 1, \
        f"not radial: {G.number_of_edges()} edges for {G.number_of_nodes()} buses"
    assert len(closed) >= 30, f"only {len(closed)} closed lines -- topology wrong"
    return ties, closed

base_net = pn.case33bw()
TIE_LINES, CLOSED_LINES = get_ties(base_net)
base_net.line.loc[TIE_LINES, "in_service"] = False
base_net.line.loc[CLOSED_LINES, "in_service"] = True

N_BUS = len(base_net.bus)
SLACK = int(base_net.ext_grid.iloc[0]["bus"])

print(f"Buses {N_BUS} | branches {len(base_net.line)} | slack bus {SLACK}")
print(f"Sectionalizing (closed): {len(CLOSED_LINES)} | tie-switches: {len(TIE_LINES)}")
print("Topology validated: radial spanning tree, all buses reachable.")
print("TIE_LINES   :", TIE_LINES)
print("CLOSED_LINES:", CLOSED_LINES)
print("Base load: %.4f MW, %.4f MVar" % (base_net.load['p_mw'].sum(),
                                          base_net.load['q_mvar'].sum()))

BASE_P = base_net.load["p_mw"].values.copy()
BASE_Q = base_net.load["q_mvar"].values.copy()

# PV generators
PV_BUSES = [6, 12, 18, 24, 30]
PV_EACH  = PV_PENETRATION * BASE_P.sum() / len(PV_BUSES)
for b in PV_BUSES:
    pp.create_sgen(base_net, bus=b, p_mw=0.0, q_mvar=0.0, name=f"PV_{b}", type="PV")
print(f"PV at buses {PV_BUSES}, {PV_EACH:.4f} MW each")

In [ ]:
def run_pf(net):
    for algo in ("bfsw", "nr"):
        try:
            pp.runpp(net, algorithm=algo, init="auto",
                     calculate_voltage_angles=False, check_connectivity=True)
            return True
        except Exception:
            continue
    return False

# Thermal limits from base-case currents plus margin
tmp = copy.deepcopy(base_net)
if run_pf(tmp) and "i_ka" in tmp.res_line.columns:
    base_net.line["max_i_ka"] = np.maximum(tmp.res_line["i_ka"].fillna(0.0) * 1.20, 0.05)
else:
    base_net.line["max_i_ka"] = 0.4
print("Base min voltage: %.4f pu" % tmp.res_bus['vm_pu'].min())
print("Line limits set. Range %.3f - %.3f kA" % (base_net.line['max_i_ka'].min(),
                                                  base_net.line['max_i_ka'].max()))
# Reference voltage floor, derived from the base case rather than assumed.
V_FLOOR = max(VMIN_ABS, float(tmp.res_bus["vm_pu"].min()) - V_TOLERANCE)
print("Base-case min voltage: %.4f pu" % tmp.res_bus["vm_pu"].min())
print("Safety floor V_FLOOR : %.4f pu" % V_FLOOR)
print("(A fixed 0.95 floor would reject the healthy base case outright.)")


## 2. Section labels and adjacency

Sections are built by hop-distance from the substation plus branch structure, so they correspond to something electrically meaningful rather than to arbitrary index ranges.

In [ ]:
G_radial = nx.Graph()
G_radial.add_nodes_from(int(b) for b in base_net.bus.index)
for lid in CLOSED_LINES:
    r = base_net.line.loc[lid]
    G_radial.add_edge(int(r["from_bus"]), int(r["to_bus"]), line_id=int(lid))

depth = nx.single_source_shortest_path_length(G_radial, SLACK)
MAX_DEPTH = max(depth.values())

def line_to_section(line_id):
    """Map a faulted line to a coarse section label (0 = no fault).
    Uses depth of the downstream end. Sections 1-3 = main feeder by
    distance; 4-7 = lateral groups. Inspect and adjust after printing."""
    if line_id < 0:
        return 0
    r = base_net.line.loc[line_id]
    f, t = int(r["from_bus"]), int(r["to_bus"])
    d = max(depth.get(f, 0), depth.get(t, 0))
    downstream = max(f, t)
    is_lateral = downstream > 17          # laterals in the Baran-Wu numbering
    if not is_lateral:
        if d <= MAX_DEPTH / 3:   return 1
        if d <= 2 * MAX_DEPTH / 3: return 2
        return 3
    if downstream <= 21: return 4
    if downstream <= 24: return 5
    if downstream <= 28: return 6
    return 7

N_SECTIONS = 8
sec_map = pd.DataFrame({
    "line_id": CLOSED_LINES,
    "from_bus": [int(base_net.line.loc[l, "from_bus"]) for l in CLOSED_LINES],
    "to_bus":   [int(base_net.line.loc[l, "to_bus"]) for l in CLOSED_LINES],
    "section":  [line_to_section(l) for l in CLOSED_LINES],
})
print(sec_map.to_string(index=False))
print("\nLines per section:"); print(sec_map["section"].value_counts().sort_index())
print("\nCheck this table. Any section with 1-2 lines will be hard to learn;")
print("any with 10+ will dominate. Adjust line_to_section if badly unbalanced.")
# Which lines have NO sensor downstream of them? Opening those does not
# zero out any measured bus, so the model cannot cheat.
ALL_SENSORS = sorted(set(sum(SENSOR_LAYOUTS.values(), [])))

def downstream_buses(line_id):
    r = base_net.line.loc[line_id]
    f, t = int(r["from_bus"]), int(r["to_bus"])
    child = t if depth.get(t, 0) > depth.get(f, 0) else f
    H = G_radial.copy()
    H.remove_edge(f, t)
    return nx.node_connected_component(H, child)

HIDDEN_LINES = []
for l in CLOSED_LINES:
    if not (downstream_buses(l) & set(ALL_SENSORS)):
        HIDDEN_LINES.append(int(l))

print("Sensor buses used anywhere:", ALL_SENSORS)
print("Lines with NO sensor downstream (hidden faults):", HIDDEN_LINES)
print("Count: %d of %d closed lines" % (len(HIDDEN_LINES), len(CLOSED_LINES)))
print()
if len(HIDDEN_LINES) < 8:
    print("WARNING: too few hidden lines for a multi-class task.")
    print("Set HIDDEN_FAULTS_ONLY = False, or thin out the S14 layout")
    print("so more of the feeder is genuinely unobserved.")

FAULT_POOL = HIDDEN_LINES if HIDDEN_FAULTS_ONLY else CLOSED_LINES
print("Sampling faults from %d lines." % len(FAULT_POOL))
print("Sections represented:", sorted({line_to_section(l) for l in FAULT_POOL}))
assert len(FAULT_POOL) >= 20, "fault pool too small -- check topology"
assert len({line_to_section(l) for l in FAULT_POOL}) >= 6, "too few sections"

# How ambiguous is localization for each layout? Two lines are confusable
# if they de-energise the same sensors but sit in different sections.
print("\nLocalization ambiguity by layout:")
for name, S in SENSOR_LAYOUTS.items():
    sig = {}
    for l in FAULT_POOL:
        key = tuple(sorted(downstream_buses(l) & set(S)))
        sig.setdefault(key, set()).add(line_to_section(l))
    amb = sum(len(v) - 1 for v in sig.values() if len(v) > 1)
    print(f"  {name:>3}: {len(sig)} distinct sensor patterns, "
          f"{amb} section(s) not separable by dead sensors alone")
print("\nHigher ambiguity at S6 is expected -- that is where topology")
print("should help, and where your hypothesis is actually tested.")


In [ ]:
# Adjacency. Ties included: topology is a fixed structural prior, identical
# in every scenario. Fault information lives in node features, not in A.
# State this explicitly in your Methodology -- reviewers will ask.
A = np.zeros((N_BUS, N_BUS), dtype=np.float32)
for _, r in base_net.line.iterrows():
    A[int(r["from_bus"]), int(r["to_bus"])] = 1.0
    A[int(r["to_bus"]), int(r["from_bus"])] = 1.0
A = A + np.eye(N_BUS, dtype=np.float32)
d = A.sum(1)
A_norm = (np.diag(1.0/np.sqrt(d+1e-8)) @ A @ np.diag(1.0/np.sqrt(d+1e-8))).astype(np.float32)
print("Adjacency built:", A_norm.shape)

## 3. Scenario mechanics

PV categories share a mean of 0.5 and differ only in spread — so a PV-category effect is a *variability* effect.

In [ ]:
PV_SPREAD = {"low": 0.10, "medium": 0.25, "high": 0.50}

def apply_load_pv(net, rng):
    load_scale = rng.uniform(0.60, 1.40)
    jitter = rng.uniform(0.90, 1.10, size=len(net.load))
    net.load["p_mw"]   = BASE_P * load_scale * jitter
    net.load["q_mvar"] = BASE_Q * load_scale * jitter
    cat = rng.choice(["low", "medium", "high"])
    pv_scale = float(np.clip(rng.normal(0.5, PV_SPREAD[cat]), 0.0, 1.0))
    if len(net.sgen) > 0:
        net.sgen["p_mw"] = PV_EACH * pv_scale
        net.sgen["q_mvar"] = 0.0
    return load_scale, pv_scale, cat

def energised_buses(net):
    G = nx.Graph(); G.add_nodes_from(int(b) for b in net.bus.index)
    for _, r in net.line.iterrows():
        if bool(r["in_service"]):
            G.add_edge(int(r["from_bus"]), int(r["to_bus"]))
    return nx.node_connected_component(G, SLACK) if SLACK in G else set()

def served_load_percent(net):
    sup = energised_buses(net)
    tot = net.load["p_mw"].sum()
    if tot <= 1e-9: return 100.0
    return 100.0 * net.load[net.load["bus"].isin(sup)]["p_mw"].sum() / tot

def safety_flags(net, v_floor=None):
    """Unsafe if energised voltage falls below the reference floor or
    rises above VMAX, or if any line exceeds its thermal rating."""
    if v_floor is None:
        v_floor = V_FLOOR
    if not hasattr(net, "res_bus") or len(net.res_bus) == 0:
        return 1, 1
    sup = energised_buses(net)
    v = net.res_bus.loc[net.res_bus.index.isin(sup), "vm_pu"]
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    vv = 1 if len(v) == 0 else int((v.min() < v_floor) or (v.max() > VMAX))
    ov = 0
    if hasattr(net, "res_line") and "loading_percent" in net.res_line.columns:
        ld = net.res_line["loading_percent"].replace([np.inf,-np.inf], np.nan).dropna()
        ov = int(len(ld) > 0 and ld.max() > MAX_LOADING)
    return vv, ov


In [ ]:
def oracle_restoration(net, faulted_line):
    """Try each tie-switch; keep the safe action restoring the most load.
    Mutates in_service in place and restores it -- no deepcopy, ~10x faster."""
    if faulted_line < 0:
        return -1, 100.0, 0, 0
    original = net.line["in_service"].copy()
    best_tie, best_load, best_vv, best_ov = -1, 0.0, 1, 1

    # Option 0: no switching
    if run_pf(net):
        r = served_load_percent(net); vv, ov = safety_flags(net)
        if vv == 0 and ov == 0:
            best_load, best_vv, best_ov = r, vv, ov

    for tie in TIE_LINES:
        net.line["in_service"] = original.values
        net.line.at[faulted_line, "in_service"] = False
        net.line.at[tie, "in_service"] = True
        if not run_pf(net):
            continue
        r = served_load_percent(net); vv, ov = safety_flags(net)
        if vv == 0 and ov == 0 and r > best_load:
            best_tie, best_load, best_vv, best_ov = int(tie), r, vv, ov

    net.line["in_service"] = original.values
    run_pf(net)                      # leave net in its post-fault state
    return best_tie, best_load, best_vv, best_ov

## 4. Sparse, noisy, missing measurements — 5 channels

| ch | meaning |
|---|---|
| 0 | voltage pu (0 if unmeasured/missing) |
| 1 | active power |
| 2 | reactive power |
| 3 | 1 if a sensor exists at this bus |
| 4 | 1 if that sensor reported a value |

A de-energised bus reads `v=0, ch3=1, ch4=1` — measured, and genuinely zero. A dropout reads `ch3=1, ch4=0`. An unmonitored bus reads `ch3=0, ch4=0`. All three are now distinguishable.

In [ ]:
N_FEAT = 5

def make_features(net, sensor_buses, noise, p_missing, rng):
    X = np.zeros((N_BUS, N_FEAT), dtype=np.float32)
    if hasattr(net, "res_bus") and len(net.res_bus) > 0:
        v = net.res_bus["vm_pu"].replace([np.inf,-np.inf], np.nan).fillna(0).values
        p = net.res_bus["p_mw"].replace([np.inf,-np.inf], np.nan).fillna(0).values
        q = net.res_bus["q_mvar"].replace([np.inf,-np.inf], np.nan).fillna(0).values
    else:
        v = p = q = np.zeros(N_BUS)
    missing = np.zeros(N_BUS, dtype=int)
    for b in sensor_buses:
        X[b, 3] = 1.0                                   # sensor exists
        if rng.random() < p_missing:
            missing[b] = 1
            continue                                    # ch4 stays 0
        X[b, 0] = v[b] + rng.normal(0, noise)
        X[b, 1] = p[b] + rng.normal(0, noise * max(abs(p[b]), 0.01))
        X[b, 2] = q[b] + rng.normal(0, noise * max(abs(q[b]), 0.01))
        X[b, 4] = 1.0                                   # value valid
    return X, missing

## 5. Generate the dataset

Time 300 scenarios and extrapolate before attempting 10,000.

In [ ]:
def generate(n_scenarios):
    rng = np.random.default_rng(SEED)
    X_list, meta, bus_rows = [], [], []
    t0 = time.time()
    for sid in range(n_scenarios):
        net = copy.deepcopy(base_net)
        load_scale, pv_scale, cat = apply_load_pv(net, rng)
        layout = str(rng.choice(list(SENSOR_LAYOUTS.keys())))
        sensors = SENSOR_LAYOUTS[layout]
        noise   = float(rng.choice(NOISE_LEVELS))
        pmiss   = float(rng.choice(MISSING_PROBS))

        fault = int(rng.random() < FAULT_PROB)
        if fault:
            fline = int(rng.choice(FAULT_POOL))
            net.line.at[fline, "in_service"] = False
            fsec = line_to_section(fline)
        else:
            fline, fsec = -1, 0

        ok = run_pf(net)
        X, miss = make_features(net, sensors, noise, pmiss, rng)
        tie, restored, vv, ov = oracle_restoration(net, fline) if ok else (-1, 0.0, 1, 1)

        meta.append(dict(scenario_id=sid, load_scale=load_scale, pv_scale=pv_scale,
            pv_category=cat, sensor_layout=layout, sensor_count=len(sensors),
            noise_level=noise, missing_probability=pmiss, fault_present=fault,
            faulted_line=fline, faulted_section=fsec, best_switch_action=tie,
            oracle_restored_load_percent=restored, voltage_violation=vv,
            line_overload_violation=ov, power_flow_success=int(ok)))
        for b in range(N_BUS):
            bus_rows.append(dict(scenario_id=sid, bus_id=b, voltage_pu=X[b,0],
                p_mw=X[b,1], q_mvar=X[b,2], sensor_available=int(X[b,3]),
                value_valid=int(X[b,4]), measurement_missing=int(miss[b])))
        X_list.append(X)
        if (sid+1) % 100 == 0:
            el = time.time()-t0
            print(f"{sid+1}/{n_scenarios}  {el:.1f}s  "
                  f"(projected 10k: {el/(sid+1)*10000/60:.1f} min)")
    return np.stack(X_list), pd.DataFrame(meta), pd.DataFrame(bus_rows)

X_all, meta, bus_df = generate(N_SCENARIOS)
print("\nX:", X_all.shape)
print("Fault rate: %.3f" % meta['fault_present'].mean())
print("PF success: %.3f" % meta['power_flow_success'].mean())
print("\nSection distribution:"); print(meta['faulted_section'].value_counts().sort_index())

In [ ]:
meta.to_csv("scenario_metadata.csv", index=False)
bus_df.to_csv("bus_features.csv", index=False)
pd.DataFrame([dict(from_bus=int(r["from_bus"]), to_bus=int(r["to_bus"]),
                   line_id=int(l), is_tie_line=int(l in TIE_LINES),
                   is_closed_base_case=int(l not in TIE_LINES))
              for l, r in base_net.line.iterrows()]).to_csv("graph_edges.csv", index=False)
np.savez("graph_dataset_arrays.npz", X=X_all, A=A_norm,
         y_fault=meta["fault_present"].values,
         y_section=meta["faulted_section"].values)
print("Saved 4 dataset files.")

## 6. Split and baselines

In [ ]:
yf = meta["fault_present"].values.astype(int)
ys = meta["faulted_section"].values.astype(int)
idx = np.arange(len(meta))

tr, te = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=ys)
tr, va = train_test_split(tr, test_size=0.1765, random_state=SEED, stratify=ys[tr])
print(f"train {len(tr)} | val {len(va)} | test {len(te)}")

Xtr, Xva, Xte = X_all[tr], X_all[va], X_all[te]
yftr, yfva, yfte = yf[tr], yf[va], yf[te]
ystr, ysva, yste = ys[tr], ys[va], ys[te]

In [ ]:
# Baseline 1: voltage threshold. Only buses with a VALID reading count.
def threshold_detector(X, thr=0.90):
    valid = X[:, :, 4] > 0.5
    v = np.where(valid, X[:, :, 0], 999.0)
    return (v.min(axis=1) < thr).astype(int)

thr_pred = threshold_detector(Xte)
print("THRESHOLD BASELINE\n", classification_report(yfte, thr_pred, digits=4))

In [ ]:
Xtr_f, Xva_f, Xte_f = [x.reshape(x.shape[0], -1) for x in (Xtr, Xva, Xte)]

rf_f = RandomForestClassifier(n_estimators=250, random_state=SEED,
                              class_weight="balanced", n_jobs=-1).fit(Xtr_f, yftr)
rf_f_pred = rf_f.predict(Xte_f)
print("RF FAULT DETECTION\n", classification_report(yfte, rf_f_pred, digits=4))

rf_s = RandomForestClassifier(n_estimators=300, random_state=SEED,
                              class_weight="balanced", n_jobs=-1).fit(Xtr_f, ystr)
rf_s_pred = rf_s.predict(Xte_f)
m = yste != 0
print("RF section accuracy on fault cases: %.4f" % accuracy_score(yste[m], rf_s_pred[m]))

## 7. Graph-aware model

Concat readout keeps per-node identity. `pooled` here is 33×hidden flattened, so the section head can tell *where* the anomaly is — which mean pooling could not.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(s, i, o):
        super().__init__(); s.lin = nn.Linear(i, o)
    def forward(s, x, A):
        return s.lin(torch.matmul(A, x))

class GraphAwareModel(nn.Module):
    def __init__(s, in_dim=N_FEAT, hidden=32, n_sections=N_SECTIONS, n_bus=N_BUS):
        super().__init__()
        s.g1 = GCNLayer(in_dim, hidden)
        s.g2 = GCNLayer(hidden, hidden)
        s.drop = nn.Dropout(0.2)
        s.node_compress = nn.Linear(hidden, 4)      # keeps readout size sane
        s.fault_head   = nn.Linear(n_bus*4, 2)
        s.section_head = nn.Linear(n_bus*4, n_sections)
    def forward(s, x, A):
        h = F.relu(s.g1(x, A)); h = s.drop(h)
        h = F.relu(s.g2(h, A))
        z = F.relu(s.node_compress(h)).flatten(1)   # per-node, order preserved
        return s.fault_head(z), s.section_head(z)

# Standardise channels 0-2 using TRAIN statistics only (no leakage).
# Raw p_mw/q_mvar dwarf voltage (~0.9-1.0); without this the network
# cannot see the voltage signal and collapses to predicting class 0.
valid_tr = Xtr[:, :, 4] > 0.5
MU = np.zeros(3, dtype=np.float32); SD = np.ones(3, dtype=np.float32)
for c in range(3):
    vals = Xtr[:, :, c][valid_tr]
    if vals.size:
        MU[c] = vals.mean(); SD[c] = max(vals.std(), 1e-6)
print("channel means:", np.round(MU,4), "| stds:", np.round(SD,4))

def scale(X):
    Xs = X.copy()
    valid = Xs[:, :, 4] > 0.5
    for c in range(3):
        Xs[:, :, c] = np.where(valid, (Xs[:, :, c] - MU[c]) / SD[c], 0.0)
    return Xs

Xtr, Xva, Xte = scale(Xtr), scale(Xva), scale(Xte)

# Class weights: section 0 is ~50% of data and swamps the 7 fault classes.
cnt = np.bincount(ystr, minlength=N_SECTIONS).astype(np.float32)
W_SEC = np.where(cnt > 0, len(ystr) / (np.maximum(cnt,1) * (cnt>0).sum()), 0.0)
cntf = np.bincount(yftr, minlength=2).astype(np.float32)
W_FLT = len(yftr) / (np.maximum(cntf,1) * 2)
print("section counts:", cnt.astype(int))
print("section weights:", np.round(W_SEC,3))

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
A_t = torch.tensor(A_norm).to(dev)
mk  = lambda X,a,b: TensorDataset(torch.tensor(X, dtype=torch.float32),
                                  torch.tensor(a, dtype=torch.long),
                                  torch.tensor(b, dtype=torch.long))
tl = DataLoader(mk(Xtr,yftr,ystr), batch_size=64, shuffle=True)
vl = DataLoader(mk(Xva,yfva,ysva), batch_size=128)
sl = DataLoader(mk(Xte,yfte,yste), batch_size=128)

W_SEC_t = torch.tensor(W_SEC, dtype=torch.float32).to(dev)
W_FLT_t = torch.tensor(W_FLT, dtype=torch.float32).to(dev)

model = GraphAwareModel(hidden=64).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-5)
print("Params:", sum(p.numel() for p in model.parameters()), "| device:", dev)

In [ ]:
def evaluate(m, loader):
    m.eval(); ft, fp, st, sp = [], [], [], []
    with torch.no_grad():
        for xb, a, b in loader:
            fl, sc = m(xb.to(dev), A_t)
            ft += a.tolist(); fp += fl.argmax(1).cpu().tolist()
            st += b.tolist(); sp += sc.argmax(1).cpu().tolist()
    return map(np.array, (ft, fp, st, sp))

EPOCHS, best_val, best_state = 150, -1, None
for ep in range(EPOCHS):
    model.train(); tot = 0
    for xb, a, b in tl:
        xb, a, b = xb.to(dev), a.to(dev), b.to(dev)
        opt.zero_grad()
        fl, sc = model(xb, A_t)
        loss = F.cross_entropy(fl, a, weight=W_FLT_t) + F.cross_entropy(sc, b, weight=W_SEC_t)
        loss.backward(); opt.step(); tot += loss.item()
    if (ep+1) % 5 == 0:
        ft, fp, st, sp = evaluate(model, vl)
        mm = st != 0
        vacc = accuracy_score(st[mm], sp[mm]) if mm.sum() else 0
        print(f"ep {ep+1:3d} loss {tot/len(tl):.4f} "
              f"val_fault {accuracy_score(ft,fp):.4f} val_sec {vacc:.4f}")
        if vacc > best_val:          # early-stopping snapshot
            best_val = vacc
            best_state = copy.deepcopy(model.state_dict())

if best_state: model.load_state_dict(best_state)
gft, gfp, gst, gsp = evaluate(model, sl)
print("\nGRAPH-AWARE FAULT DETECTION\n", classification_report(gft, gfp, digits=4))
mm = gst != 0
print("Graph section accuracy on fault cases: %.4f" % accuracy_score(gst[mm], gsp[mm]))

# Collapse check: a model predicting one class always is not learning.
u, c = np.unique(gsp, return_counts=True)
print("\nPredicted section distribution:", dict(zip(u.tolist(), c.tolist())))
print("True section distribution:      ",
      dict(zip(*[x.tolist() for x in np.unique(gst, return_counts=True)])))
if len(u) == 1:
    print("\nCOLLAPSED: model predicts one class only. Do not report these"
          "\nnumbers. Raise EPOCHS, raise lr, or check the scaling cell ran.")

## 8. Restoration comparison and robustness

In [ ]:
test_meta = meta.iloc[te].reset_index(drop=True)
oracle = test_meta["oracle_restored_load_percent"].values

def ai_restoration(y_true, y_pred, oracle):
    """Conservative: credit oracle load only when the section is right.
    A wrong section means switching on bad information -> score 0.
    Say plainly in the paper that this is a lower bound."""
    return np.array([100.0 if (t==0 and p==0) else (r if (t!=0 and p==t) else 0.0)
                     for t, p, r in zip(y_true, y_pred, oracle)])

rf_rest = ai_restoration(yste, rf_s_pred, oracle)
g_rest  = ai_restoration(gst, gsp, oracle)
print("Oracle       : %.2f%%" % oracle.mean())
print("RF-assisted  : %.2f%%" % rf_rest.mean())
print("Graph-assisted: %.2f%%" % g_rest.mean())

plt.figure(figsize=(6,4))
plt.bar(["Oracle","Random Forest","Graph-Aware"],
        [oracle.mean(), rf_rest.mean(), g_rest.mean()])
plt.ylabel("Average restored load (%)"); plt.ylim(0,100)
plt.title("Oracle vs AI-Assisted Restoration"); plt.grid(axis="y", alpha=0.3)
plt.savefig("restoration_comparison.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
res = test_meta.copy()
res["rf_fault_correct"]    = (rf_f_pred == yfte).astype(int)
res["graph_fault_correct"] = (gfp == gft).astype(int)
res["rf_section_correct"]    = (rf_s_pred == yste).astype(int)
res["graph_section_correct"] = (gsp == gst).astype(int)
cols = ["rf_fault_correct","graph_fault_correct","rf_section_correct","graph_section_correct"]

for key, fname in [("noise_level","performance_vs_noise"),
                   ("sensor_count","performance_vs_sensor_count"),
                   ("missing_probability","performance_vs_missing"),
                   ("pv_category","performance_vs_pv")]:
    s = res.groupby(key)[cols].mean().reset_index()
    s.to_csv(f"{fname}.csv", index=False)
    print(f"\n=== by {key} ==="); print(s.to_string(index=False))

ss = res.groupby("sensor_count")[cols].mean().reset_index()
plt.figure(figsize=(7,4))
plt.plot(ss["sensor_count"], ss["rf_section_correct"], marker="o", label="Random Forest")
plt.plot(ss["sensor_count"], ss["graph_section_correct"], marker="s", label="Graph-aware")
plt.xlabel("Number of sensor buses"); plt.ylabel("Section accuracy")
plt.title("Section Accuracy vs Sensor Count"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("performance_vs_sensor_count.png", dpi=300, bbox_inches="tight"); plt.show()

summary = pd.DataFrame({
    "Model": ["Threshold","Random Forest","Graph-Aware"],
    "Fault_Detection_Accuracy": [accuracy_score(yfte, thr_pred),
                                 accuracy_score(yfte, rf_f_pred),
                                 accuracy_score(gft, gfp)],
    "Section_Accuracy_on_Faults": [np.nan,
        accuracy_score(yste[yste!=0], rf_s_pred[yste!=0]),
        accuracy_score(gst[gst!=0], gsp[gst!=0])],
    "AI_Assisted_Restored_Load_Percent": [np.nan, rf_rest.mean(), g_rest.mean()]})
summary.to_csv("final_results_summary.csv", index=False)
print("\n", summary.to_string(index=False))

## 9. What to check before believing any of this

**Restoration.** `Oracle` should now be well above zero. If it is still 0.00%,
the safety check is still rejecting everything — print `V_FLOOR` and compare
it against the base-case minimum.

**Section accuracy.** Anything at exactly 1.000 means the task is still
trivial. Check the hidden-line count printed in Section 2: if faults are
being sampled from lines that *do* have sensors downstream, the models are
reading dead sensors rather than inferring position.

**Threshold baseline.** With `FAULT_PROB = 0.50`, a model that always
predicts "fault" scores 0.50. If the threshold rule sits near 0.50, it is
degenerate — check its precision and recall, not just accuracy.

**The result your hypothesis predicts** is in the `by sensor_count` table:
graph-aware ahead of Random Forest at 6 sensors, with the gap narrowing or
reversing at 14. That single row is the core of your paper.

If graph-aware loses everywhere, report it. "Topology did not help under
these conditions" is a real finding, and defending it honestly is worth
more than a number you tuned until it looked right.
